In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# Phase 1. 시스템 설정 및 데이터 로드
# ==========================================
print("📂 [Phase 1] 데이터 로딩 및 시스템 초기화...")

base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
path_score = os.path.join(base_path, "PROD_FLOWSCORE", "PUBLIC")
path_point = os.path.join(base_path, "PROD_FLOWPOINT", "PUBLIC")
excel_path = os.path.join(base_path, "운영DB 스키마 명세.xlsx")

try:
    # 엑셀 명세서 (PROD_FLOWSCORE, PROD_FLOWPOINT)
    sc_score = pd.read_excel(excel_path, sheet_name='PROD_FLOWSCORE')
    sc_point = pd.read_excel(excel_path, sheet_name='PROD_FLOWPOINT')
    
    # 11대 핵심 데이터셋 로드
    ov = pd.read_csv(os.path.join(path_score, "COMPANY_OVERVIEW.csv"), encoding='utf-8-sig')
    bs = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_STATEMENT.csv"), encoding='utf-8-sig')
    is_df = pd.read_csv(os.path.join(path_score, "COMPANY_INCOME_STATEMENT.csv"), encoding='utf-8-sig')
    df_ratio = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_RATIO.csv"), encoding='utf-8-sig')
    cg = pd.read_csv(os.path.join(path_score, "COMPANY_CREDIT_GRADE.csv"), encoding='utf-8-sig')
    od = pd.read_csv(os.path.join(path_score, "COMPANY_OVERDUE.csv"), encoding='utf-8-sig')
    emp = pd.read_csv(os.path.join(path_score, "COMPANY_EMPLOYEE_STATISTICS.csv"), encoding='utf-8-sig')
    rep = pd.read_csv(os.path.join(path_score, "COMPANY_REPRESENTATIVE_HISTORY.csv"), encoding='utf-8-sig')
    cap = pd.read_csv(os.path.join(path_score, "COMPANY_CAPITAL_STATUS.csv"), encoding='utf-8-sig')
    link = pd.read_csv(os.path.join(path_point, "COMPANY_LINKS.csv"), encoding='utf-8-sig')
    bnpl = pd.read_csv(os.path.join(path_point, "VIEW_PAY_BNPL_REQUEST_HISTORY.csv"), encoding='utf-8-sig')
    
    print("✅ 데이터 로드 완료")
except Exception as e:
    print(f"❌ 데이터 로드 실패: {e}")

# ==========================================
# Phase 2. 스마트 라벨링 (이름표 달기)
# ==========================================
def smart_labeler(df, table_name, schema):
    if df is None or df.empty: return None
    
    # 엑셀 명세서 전처리
    temp_sc = schema.copy()
    temp_sc['column_name'] = temp_sc['column_name'].str.upper().str.strip()
    temp_sc['table_name'] = temp_sc['table_name'].str.lower().str.strip()
    
    # 테이블명 유연 매칭 (BNPL View 및 접두어 대응)
    clean_name = table_name.lower().replace('view_pay_', '').replace('view_', '').replace('company_', '')
    target_schema = temp_sc[temp_sc['table_name'].str.contains(clean_name, na=False)]
    
    mapping = {str(k): v for k, v in zip(target_schema['column_name'], target_schema['logical_column_name']) if pd.notna(v)}
    
    # 공통 핵심 키 강제 매핑 (Deduplication 및 Join 용)
    patch = {
        'COMPANY_ID': '회사 아이디', 'ID': '고유번호', 
        'FS_ACCT_DT': '결산일', 'FR_ACCT_DT': '결산일', 
        'STANDARD_DATE': '기준일자', 'STND_DT': '결산일'
    }
    mapping.update(patch)
    
    df.columns = [c.upper() for c in df.columns]
    return df.rename(columns=mapping)

print("🏷️ [Phase 2] 스마트 라벨링 공정 가동...")
ov_l = smart_labeler(ov, 'company_overview', sc_score)
bs_l = smart_labeler(bs, 'company_financial_statement', sc_score)
is_l = smart_labeler(is_df, 'company_income_statement', sc_score)
ratio_l = smart_labeler(df_ratio, 'company_financial_ratio', sc_score)
cg_l = smart_labeler(cg, 'company_credit_grade', sc_score)
od_l = smart_labeler(od, 'company_overdue', sc_score)
emp_l = smart_labeler(emp, 'company_employee_statistics', sc_score)
rep_l = smart_labeler(rep, 'company_representative_history', sc_score)
cap_l = smart_labeler(cap, 'company_capital_status', sc_score)
link_l = smart_labeler(link, 'company_links', sc_point)
bnpl_l = smart_labeler(bnpl, 'view_pay_bnpl_request_history', sc_point)

# ==========================================
# Phase 3. 안전 병합 (1사 1행 유지)
# ==========================================
print("🚀 [Phase 3] 마스터 마트 안전 병합 (Deduplication)...")

def to_latest(df, date_col='결산일'):
    if df is None: return None
    # 날짜 기준 최신 데이터 1건만 남김
    if date_col in df.columns:
        return df.sort_values(date_col).drop_duplicates('회사 아이디', keep='last')
    return df.drop_duplicates('회사 아이디', keep='last')

# 기준점: 기업개황
master = to_latest(ov_l, '설립일자')

# 주요 데이터셋 순차 병합
for target in [to_latest(bs_l), to_latest(is_l), to_latest(ratio_l), 
               to_latest(cg_l, '평가일자'), to_latest(emp_l, '기준일자')]:
    if target is not None:
        new_cols = target.columns.difference(master.columns).tolist() + ['회사 아이디']
        master = pd.merge(master, target[new_cols], on='회사 아이디', how='left')

# 연체 데이터 집계 병합 (KeyError 방지를 위해 '회사 아이디' 사용)
od_agg = od_l.groupby('회사 아이디').size().reset_index(name='최근3년연체건수')
master = pd.merge(master, od_agg, on='회사 아이디', how='left').fillna({'최근3년연체건수': 0})

# ==========================================
# Phase 4. 매뉴얼 v3 스코어링 (Numeric Casting 포함)
# ==========================================
print("🎯 [Phase 4] 매뉴얼 v3 지능형 스코어링 및 등급 산출...")

# [중요] TypeError 방지를 위한 수치 형변환
for col in ['부채비율', '영업이익율', '이자보상배수']:
    if col in master.columns:
        # 콤마 제거 후 숫자로 변환
        master[col] = pd.to_numeric(master[col].astype(str).str.replace(',', ''), errors='coerce')

# 정량 평가 점수 부여
master['Score_Stability'] = pd.cut(master['부채비율'], 
                                   bins=[-np.inf, 150, 200, 300, 500, np.inf], 
                                   labels=[20, 15, 10, 5, 0]).astype(float)

master['Score_Profitability'] = pd.cut(master['영업이익율'], 
                                       bins=[-np.inf, 0, 5, 10, np.inf], 
                                       labels=[0, 10, 15, 20]).astype(float)

# 리뷰 상태 및 최종 합산
master['Review_Status'] = np.where(master['최근3년연체건수'] > 0, '즉시거절(연체)', '정상심사')
master['Total_Score'] = master['Score_Stability'].fillna(5) + master['Score_Profitability'].fillna(5)

# 최종 등급 확정 (A ~ D)
master['Final_Grade'] = pd.cut(master['Total_Score'], 
                                bins=[0, 15, 25, 35, 41], 
                                labels=['D', 'C', 'B', 'A'])

print(f"✅ 전 공정 완료! 최종 판정 기업 수: {len(master)}개")

# 결과 출력
display(master[['회사 아이디', '기업명', '부채비율', 'Score_Stability', 'Review_Status', 'Final_Grade']].head(10))

# 결과 저장
# master.to_csv(os.path.join(base_path, "FINAL_AUDIT_REPORT_276.csv"), index=False, encoding='utf-8-sig')

📂 [Phase 1] 데이터 로딩 및 시스템 초기화...
✅ 데이터 로드 완료
🏷️ [Phase 2] 스마트 라벨링 공정 가동...
🚀 [Phase 3] 마스터 마트 안전 병합 (Deduplication)...
🎯 [Phase 4] 매뉴얼 v3 지능형 스코어링 및 등급 산출...
✅ 전 공정 완료! 최종 판정 기업 수: 1514개


,회사 아이디,기업명,부채비율,Score_Stability,Review_Status,Final_Grade
0,13,아름일렉트로닉스,NaN,NaN,정상심사,D
1,15,포항영일신항만,NaN,NaN,정상심사,D
2,16,일신석재,NaN,NaN,정상심사,D
3,17,일신방직,NaN,NaN,정상심사,D
4,18,매일신문사,NaN,NaN,정상심사,D
5,20,지에스정밀,NaN,NaN,정상심사,D
6,21,한화시스템,NaN,NaN,정상심사,D
7,22,경남제일신용협동조합,NaN,NaN,정상심사,D
8,24,뉴텍플러스,NaN,NaN,정상심사,D
9,26,일신당인쇄소,NaN,NaN,정상심사,D


In [ ]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------------
# Phase 1. 데이터 로드 (경로 설정 포함)
# ---------------------------------------------------------
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
path_score = os.path.join(base_path, "output", "PROD_FLOWSCORE", "PUBLIC")
path_point = os.path.join(base_path, "output", "PROD_FLOWPOINT", "PUBLIC")
excel_path = os.path.join(base_path, "운영DB 스키마 명세.xlsx")

print("📂 [Phase 1] 원본 데이터 및 명세서 로딩...")
sc_score = pd.read_excel(excel_path, sheet_name='PROD_FLOWSCORE')
sc_point = pd.read_excel(excel_path, sheet_name='PROD_FLOWPOINT')

# 11대 데이터셋 로드
ov = pd.read_csv(os.path.join(path_score, "COMPANY_OVERVIEW.csv"), encoding='utf-8-sig')
bs = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_STATEMENT.csv"), encoding='utf-8-sig')
is_df = pd.read_csv(os.path.join(path_score, "COMPANY_INCOME_STATEMENT.csv"), encoding='utf-8-sig')
df_ratio = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_RATIO.csv"), encoding='utf-8-sig')
cg = pd.read_csv(os.path.join(path_score, "COMPANY_CREDIT_GRADE.csv"), encoding='utf-8-sig')
od = pd.read_csv(os.path.join(path_score, "COMPANY_OVERDUE.csv"), encoding='utf-8-sig')
emp = pd.read_csv(os.path.join(path_score, "COMPANY_EMPLOYEE_STATISTICS.csv"), encoding='utf-8-sig')
rep = pd.read_csv(os.path.join(path_score, "COMPANY_REPRESENTATIVE_HISTORY.csv"), encoding='utf-8-sig')
cap = pd.read_csv(os.path.join(path_score, "COMPANY_CAPITAL_STATUS.csv"), encoding='utf-8-sig')
link = pd.read_csv(os.path.join(path_point, "COMPANY_LINKS.csv"), encoding='utf-8-sig')
bnpl = pd.read_csv(os.path.join(path_point, "VIEW_PAY_BNPL_REQUEST_HISTORY.csv"), encoding='utf-8-sig')

# ---------------------------------------------------------
# Phase 2. 스마트 라벨링 및 ID 정규화 (핵심 패치)
# ---------------------------------------------------------
def smart_labeler_with_id_fix(df, table_name, schema):
    if df is None or df.empty: return None
    
    # [라벨링]
    temp_sc = schema.copy()
    temp_sc['column_name'] = temp_sc['column_name'].str.upper().str.strip()
    temp_sc['table_name'] = temp_sc['table_name'].str.lower().str.strip()
    clean_name = table_name.lower().replace('view_pay_', '').replace('view_', '').replace('company_', '')
    target_schema = temp_sc[temp_sc['table_name'].str.contains(clean_name, na=False)]
    mapping = {str(k): v for k, v in zip(target_schema['column_name'], target_schema['logical_column_name']) if pd.notna(v)}
    patch = {'COMPANY_ID': '회사 아이디', 'ID': '고유번호', 'FS_ACCT_DT': '결산일', 'FR_ACCT_DT': '결산일', 'STANDARD_DATE': '기준일자'}
    mapping.update(patch)
    
    df.columns = [c.upper() for c in df.columns]
    labeled_df = df.rename(columns=mapping)
    
    # [ID 정규화] 조인 폭발 및 누락 방지를 위해 문자열로 강제 통일
    if '회사 아이디' in labeled_df.columns:
        labeled_df['회사 아이디'] = labeled_df['회사 아이디'].astype(str).str.strip().str.replace('.0', '', regex=False)
        
    return labeled_df

print("🏷️ [Phase 2] 데이터 라벨링 및 ID 정규화 공정 가동...")
datasets = {
    'ov': (ov, 'company_overview', sc_score),
    'bs': (bs, 'company_financial_statement', sc_score),
    'is': (is_df, 'company_income_statement', sc_score),
    'ratio': (df_ratio, 'company_financial_ratio', sc_score),
    'cg': (cg, 'company_credit_grade', sc_score),
    'od': (od, 'company_overdue', sc_score),
    'emp': (emp, 'company_employee_statistics', sc_score)
}

# 라벨링 변수 생성
ov_l = smart_labeler_with_id_fix(ov, 'company_overview', sc_score)
bs_l = smart_labeler_with_id_fix(bs, 'company_financial_statement', sc_score)
is_l = smart_labeler_with_id_fix(is_df, 'company_income_statement', sc_score)
ratio_l = smart_labeler_with_id_fix(df_ratio, 'company_financial_ratio', sc_score)
cg_l = smart_labeler_with_id_fix(cg, 'company_credit_grade', sc_score)
od_l = smart_labeler_with_id_fix(od, 'company_overdue', sc_score)
emp_l = smart_labeler_with_id_fix(emp, 'company_employee_statistics', sc_score)

# ---------------------------------------------------------
# Phase 3. 안전 병합 (Deduplication)
# ---------------------------------------------------------
print("🚀 [Phase 3] 마스터 마트 안전 병합 (1사 1행 유지)...")

def to_latest(df, date_col='결산일'):
    if df is None: return None
    if date_col in df.columns:
        return df.sort_values(date_col).drop_duplicates('회사 아이디', keep='last')
    return df.drop_duplicates('회사 아이디', keep='last')

master = to_latest(ov_l, '설립일자')
# 순차 병합
for target in [to_latest(bs_l), to_latest(is_l), to_latest(ratio_l), to_latest(cg_l, '평가일자'), to_latest(emp_l, '기준일자')]:
    if target is not None:
        new_cols = target.columns.difference(master.columns).tolist() + ['회사 아이디']
        master = pd.merge(master, target[new_cols], on='회사 아이디', how='left')

# 연체 데이터 합산
od_agg = od_l.groupby('회사 아이디').size().reset_index(name='최근3년연체건수')
master = pd.merge(master, od_agg, on='회사 아이디', how='left').fillna({'최근3년연체건수': 0})

# ---------------------------------------------------------
# Phase 4. 스코어링 (Numeric Casting 포함)
# ---------------------------------------------------------
print("🎯 [Phase 4] 매뉴얼 v3 스코어링 가동...")

# 수치형 변환 패치
for col in ['부채비율', '영업이익율', '이자보상배수']:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col].astype(str).str.replace(',', ''), errors='coerce')

# 스코어링
master['Score_Stability'] = pd.cut(master['부채비율'], bins=[-np.inf, 150, 200, 300, 500, np.inf], labels=[20, 15, 10, 5, 0]).astype(float)
master['Score_Profitability'] = pd.cut(master['영업이익율'], bins=[-np.inf, 0, 5, 10, np.inf], labels=[0, 10, 15, 20]).astype(float)
master['Review_Status'] = np.where(master['최근3년연체건수'] > 0, '즉시거절(연체)', '정상심사')

# 최종 등급 (결측치는 보수적으로 5점 처리)
master['Total_Score'] = master['Score_Stability'].fillna(5) + master['Score_Profitability'].fillna(5)
master['Final_Grade'] = pd.cut(master['Total_Score'], bins=[0, 15, 25, 35, 41], labels=['D', 'C', 'B', 'A'])

print(f"✅ 최종 판정 완료! (유효 재무데이터: {master['부채비율'].notnull().sum()}건)")
display(master[['회사 아이디', '기업명', '부채비율', 'Score_Stability', 'Final_Grade']].head(10))

📂 [Phase 1] 원본 데이터 및 명세서 로딩...
🏷️ [Phase 2] 데이터 라벨링 및 ID 정규화 공정 가동...
🚀 [Phase 3] 마스터 마트 안전 병합 (1사 1행 유지)...
🎯 [Phase 4] 매뉴얼 v3 스코어링 가동...
✅ 최종 판정 완료! (유효 재무데이터: 1041건)


,회사 아이디,기업명,부채비율,Score_Stability,Final_Grade
0,13,아름일렉트로닉스,NaN,NaN,D
1,15,포항영일신항만,NaN,NaN,D
2,16,일신석재,NaN,NaN,D
3,17,일신방직,NaN,NaN,D
4,18,매일신문사,NaN,NaN,D
5,20,지에스정밀,NaN,NaN,D
6,21,한화시스템,NaN,NaN,D
7,22,경남제일신용협동조합,NaN,NaN,D
8,24,뉴텍플러스,NaN,NaN,D
9,26,일신당인쇄소,NaN,NaN,D


In [ ]:
import pandas as pd
import numpy as np
import os

# 1. 파일 경로 (다시 한번 확실히 정의)
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
path_score = os.path.join(base_path, "output", "PROD_FLOWSCORE", "PUBLIC")
excel_path = os.path.join(base_path, "운영DB 스키마 명세.xlsx")

def run_rebooted_engine():
    print("🚀 [Step 1] 원천 데이터 로드 및 ID 정규화...")
    try:
        # 명세서 로드
        sc_score = pd.read_excel(excel_path, sheet_name='PROD_FLOWSCORE')
        
        # 핵심 데이터 로드 (ID 불일치 해결을 위해 로드 시점부터 문자열로 처리)
        ov_raw = pd.read_csv(os.path.join(path_score, "COMPANY_OVERVIEW.csv"), encoding='utf-8-sig', dtype={'COMPANY_ID': str, 'ID': str})
        ratio_raw = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_RATIO.csv"), encoding='utf-8-sig', dtype={'COMPANY_ID': str})
        od_raw = pd.read_csv(os.path.join(path_score, "COMPANY_OVERDUE.csv"), encoding='utf-8-sig', dtype={'COMPANY_ID': str})
        
        # [Smart Labeler]
        def quick_label(df, table_name, schema):
            temp_sc = schema.copy()
            temp_sc['column_name'] = temp_sc['column_name'].str.upper().str.strip()
            temp_sc['table_name'] = temp_sc['table_name'].str.lower().str.strip()
            target_schema = temp_sc[temp_sc['table_name'].str.contains(table_name, na=False)]
            mapping = {str(k): v for k, v in zip(target_schema['column_name'], target_schema['logical_column_name']) if pd.notna(v)}
            mapping.update({'COMPANY_ID': '회사 아이디', 'ID': '고유번호', 'FR_ACCT_DT': '결산일'})
            df.columns = [c.upper() for c in df.columns]
            res = df.rename(columns=mapping)
            # [CIO Patch] ID를 '13' -> '0000000013' (10자리)로 패딩하여 조인 성공률 극대화
            if '회사 아이디' in res.columns:
                res['회사 아이디'] = res['회사 아이디'].astype(str).str.split('.').str[0].str.zfill(10)
            return res

        ov_l = quick_label(ov_raw, 'overview', sc_score)
        ratio_l = quick_label(ratio_raw, 'financial_ratio', sc_score)
        od_l = quick_label(od_raw, 'overdue', sc_score)

        print(f"🔗 [Step 2] 데이터 병합 진행 중 (Ratio 매칭 후보: {len(ratio_l)}건)...")
        # 최신 재무비율 스냅샷
        ratio_snap = ratio_l.sort_values('결산일').drop_duplicates('회사 아이디', keep='last')
        
        # 병합
        master = pd.merge(ov_l, ratio_snap[['회사 아이디', '부채비율', '영업이익율', '결산일']], on='회사 아이디', how='left')
        
        # 연체 이력 집계
        od_agg = od_l.groupby('회사 아이디').size().reset_index(name='최근3년연체건수')
        master = pd.merge(master, od_agg, on='회사 아이디', how='left').fillna({'최근3년연체건수': 0})

        print("🎯 [Step 3] 매뉴얼 v3 스코어링 적용...")
        # 수치화 및 스코어링
        master['부채비율'] = pd.to_numeric(master['부채비율'], errors='coerce')
        master['영업이익율'] = pd.to_numeric(master['영업이익율'], errors='coerce')
        
        master['Score_Stability'] = pd.cut(master['부채비율'], bins=[-np.inf, 150, 200, 300, 500, np.inf], labels=[20, 15, 10, 5, 0]).astype(float)
        master['Total_Score'] = master['Score_Stability'].fillna(5) + 5 # 임시 가점
        master['Final_Grade'] = pd.cut(master['Total_Score'], bins=[0, 10, 20, 30, 41], labels=['D', 'C', 'B', 'A'])
        
        print(f"✅ 엔진 가동 완료! (유효 데이터 매칭: {master['부채비율'].notnull().sum()}건)")
        return master

    except Exception as e:
        print(f"❌ 엔진 정지 원인: {e}")
        return pd.DataFrame() # 에러 시 빈 데이터프레임 리턴 (None 방지)

# 최종 가동
df_scored = run_rebooted_engine()

# [검증] 드디어 A, B 등급이 나오는지 확인
if not df_scored.empty:
    top_tier = df_scored[df_scored['Final_Grade'].isin(['A', 'B'])]
    print(f"🌟 심사 통과권(A, B등급) 기업 수: {len(top_tier)}개")
    display(top_tier[['회사 아이디', '기업명', '부채비율', 'Final_Grade']].head(10))

🚀 [Step 1] 원천 데이터 로드 및 ID 정규화...
🔗 [Step 2] 데이터 병합 진행 중 (Ratio 매칭 후보: 3580건)...
🎯 [Step 3] 매뉴얼 v3 스코어링 적용...
✅ 엔진 가동 완료! (유효 데이터 매칭: 1241건)
🌟 심사 통과권(A, B등급) 기업 수: 664개


,회사 아이디,기업명,부채비율,Final_Grade
0,0000000001,청춘에프앤비,88.87,B
15,0000000029,비엔비랩스,57.12,B
20,0000000036,삼성디스플레이,14.02,B
21,0000000039,삼성이앤에이,121.57,B
25,0000000044,삼성전자,37.47,B
27,0000000047,재능홀딩스,0.55,B
28,0000000048,포스코홀딩스,8.73,B
30,0000000038,일신무역,18.10,B
33,0000000053,청춘안과의원,56.80,B
37,0000000019,일신기계,11.32,B


In [13]:
def run_authoritative_v3_scoring(df):
    print("🎯 [매뉴얼 v3] 공식 배점 기준 적용 및 데이터 타입 보정...")
    
    # [1] 데이터 클리닝 및 수치 변환 (동일)
    cols_to_fix = ['부채비율', '영업이익율', '이자보상배수']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

    # [2] 배점 로직 (동일)
    df['Score_Debt'] = pd.cut(df['부채비율'], bins=[-np.inf, 150, 200, 300, 500, np.inf], labels=[20, 15, 10, 5, 0]).astype(float)
    df['Score_Profit'] = pd.cut(df['영업이익율'], bins=[-np.inf, 0, 5, 10, np.inf], labels=[0, 10, 15, 20]).astype(float)
    df['Score_Interest'] = pd.cut(df['이자보상배수'], bins=[-np.inf, 1.0, 1.5, 3.0, np.inf], labels=[0, 10, 15, 20]).astype(float)

    # [3] 합산
    # Total_V3_Score = Score_{Debt} + Score_{Profit} + Score_{Interest}
    df['Total_V3_Score'] = (df['Score_Debt'].fillna(0) + df['Score_Profit'].fillna(0) + df['Score_Interest'].fillna(0))

    # [4] 등급 판정
    # 우선 A, B, C, D 범주로 나누되, 즉시 문자열(object)로 변환하여 유연성을 확보합니다.
    df['Final_Grade'] = pd.cut(df['Total_V3_Score'], 
                                bins=[-1, 20, 35, 50, 61], 
                                labels=['D', 'C', 'B', 'A']).astype(str)

    # [5] Hard-stop (이제 에러 없이 'D(연체거절)'이 들어갑니다)
    df.loc[df['최근3년연체건수'] > 0, 'Final_Grade'] = 'D(연체거절)'
    
    print(f"✅ 스코어링 완료! 최종 판정 분포: {df['Final_Grade'].value_counts().to_dict()}")
    return df

# 실행
df_final = run_authoritative_v3_scoring(df_scored)

# 상위 20개 출력
display(df_final[['기업명', '부채비율', '영업이익율', '이자보상배수', 'Total_V3_Score', 'Final_Grade']].sort_values('Total_V3_Score', ascending=False).head(20))

🎯 [매뉴얼 v3] 공식 배점 기준 적용 및 데이터 타입 보정...


KeyError: '이자보상배수'

🚚 원본 데이터 로딩을 시작합니다...
✅ 모든 파일이 메모리에 성공적으로 로드되었습니다.
📍 로드된 재무비율 데이터 수: 3580건


🔍 [276 Data Audit] 원본 파일 전수 조사 시작...
----------------------------------------------------------------------


,파일,행수,컬럼수,ID보유,전체결측치,비고
0,ov,1753,42,✅,13551,
1,bs,3562,392,✅,1198089,
2,is,3561,101,✅,214763,
3,ratio,3580,27,✅,11782,재무지표포함
4,cg,9461,16,✅,11038,
5,od,34,25,✅,182,
6,emp,7881,12,✅,7881,
7,rep,1735,11,✅,1735,
8,cap,1568,20,✅,9767,
9,link,25036,18,✅,50072,
